# Forecaster — full walkthrough

Everything you need to actually *use* this platform from a notebook:
the SDK surface, how to bring in new datasets, how to compute custom
features, how to record results so they show up on the dashboard, and
how to query run history.

**Mental model:**
- `data/warehouse.db` is a single SQLite file. Both this notebook and the
  Django dashboard read/write the same file.
- The **tracker SDK** writes via raw `sqlite3` (zero Django dependency in
  notebooks). The **dashboard** reads via Django ORM. Both see the same data.
- You never need to restart anything. Run a cell → refresh the browser → see it.

**Tables you'll see:**

| Layer | Tables |
|---|---|
| Raw inputs | `raw_prices`, `raw_factors`, `dim_symbols` |
| Features | `fct_features` (the supervised dataset; `target_1m` is what you predict) |
| Tracker | `experiment`, `run`, `run_param`, `run_metric`, `run_tag`, `run_artifact`, `run_feature_importance` |
| Strategy outputs | `rf_predictions`, `rf_picks`, `rf_portfolio_returns`, `rf_portfolio_weights`, `perm_summary` |
| DQ + ingestion history | `dq_runs`, `dq_checks`, `ingest_runs` |

## 1. Connect to the warehouse

The SDK auto-discovers `data/warehouse.db` by walking up from your
notebook's working directory. Override with `FORECASTER_DB_PATH` if needed.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import tracker
from tracker import resolve_warehouse_path

DB = resolve_warehouse_path()
print('warehouse:', DB)

con = sqlite3.connect(DB)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)
print(f'\n{len(tables)} tables:')
print('  ' + '\n  '.join(tables['name'].tolist()))

## 2. Explore what's already there

Before doing anything new, see what data the platform has already prepared.

In [ ]:
# What symbols and how much price history?
pd.read_sql("""
  SELECT s.symbol, s.sector, MIN(p.date) AS first_date, MAX(p.date) AS last_date,
         COUNT(*) AS n_rows
  FROM dim_symbols s JOIN raw_prices p ON p.symbol = s.symbol
  GROUP BY s.symbol, s.sector
  ORDER BY n_rows DESC
""", con)

In [ ]:
# fct_features is the supervised dataset. target_1m = next-month return per ticker.
feats = pd.read_sql(
    "SELECT * FROM fct_features WHERE target_1m IS NOT NULL ORDER BY date, symbol",
    con,
    parse_dates=['date'],
)
feats.shape, feats.columns.tolist()

In [ ]:
feats.head(8)

## 3. The tracker SDK — full surface

Everything is one of these calls:

```python
with tracker.run(
    experiment="my_exp",         # required — groups runs in the dashboard
    params={...},                # initial param dict (any JSON-serialisable values)
    tags=["foo", "bar"],         # initial tags
    name="baseline_v1",          # optional human-friendly label
    description="...",           # only set on first creation of the experiment
) as r:
    r.param(key, value)              # add one more param mid-run
    r.params({...})                  # bulk add
    r.log_metric("oos_sharpe", 1.2)  # single metric
    r.log_metric("loss", v, step=epoch)   # step-indexed series
    r.log_metrics({"sharpe": 1.2, "mae": 0.02})   # bulk
    r.tag("another_tag")             # add a tag
    r.log_importance(dict_series_or_2col_df)   # ranked feature importance
    r.log_artifact("name", "/path/to/file", kind="plot")

# Outside the with-block: status flips to 'completed' (or 'failed' if exception).
# Then the run appears on http://localhost:5173/  (refresh the leaderboard).

# Append more to a finished run later (e.g. portfolio metrics post-backtest):
with tracker.attach(run_id) as r:
    r.log_metric("new_metric", 0.99)
    r.tag("reanalysed")
```

## 4. A minimal experiment

Train a tiny ridge regression on the existing features, evaluate OOS,
and log the result. This is the smallest end-to-end thing you can do.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

FEATURES = [
    "return_1m", "return_3m", "return_12m",
    "rsi", "atr", "macd",
    "beta_mkt_rf", "beta_smb", "beta_hml",
]
df = feats.dropna(subset=FEATURES + ["target_1m"]).sort_values("date")

# Walk-forward-ish: chronological 80/20 split (keeps it brief; in real research use
# the proper expanding-window split that models_rf.strategy uses).
split_at = df["date"].quantile(0.8)
train = df[df["date"] <= split_at]
test  = df[df["date"] >  split_at]

scaler = StandardScaler().fit(train[FEATURES])
Xtr = scaler.transform(train[FEATURES]); ytr = train["target_1m"]
Xte = scaler.transform(test[FEATURES]);  yte = test["target_1m"]

model = Ridge(alpha=1.0, random_state=0).fit(Xtr, ytr)
y_pred = model.predict(Xte)

oos_mae = float(mean_absolute_error(yte, y_pred))
oos_r2  = float(r2_score(yte, y_pred))
ic      = float(np.corrcoef(yte, y_pred)[0, 1])  # information coefficient

print(f"n_train={len(train):>5}  n_test={len(test):>5}")
print(f"oos_mae = {oos_mae:.5f}")
print(f"oos_r2  = {oos_r2:+.5f}")
print(f"ic      = {ic:+.4f}")

In [ ]:
# Now log it as a run. After this cell finishes, refresh the browser.
with tracker.run(
    experiment="notebook_walkthrough",
    params={
        "model": "ridge",
        "alpha": 1.0,
        "features": FEATURES,
        "split_method": "time_quantile_80",
        "n_train": int(len(train)),
        "n_test": int(len(test)),
    },
    tags=["ridge", "notebook"],
    name="ridge_alpha1",
) as r:
    r.log_metric("oos_mae", oos_mae)
    r.log_metric("oos_r2", oos_r2)
    r.log_metric("ic", ic)
    # Importance via |coef|: meaningful for linear models.
    r.log_importance(dict(zip(FEATURES, np.abs(model.coef_))))
    print("run_id:", r.run_id)

# Open http://localhost:5173 — the run is now on the leaderboard.

## 5. Adding a new dataset

Two paths.

**A. Free-form research table** (lightweight, no migrations).
Use `pandas.to_sql` to drop your data into the warehouse as a new table.
Other notebooks can read it back. Won't show up in the dashboard but is
instantly available.

**B. Tracked ingestion** (proper, replayable).
Add a Django model + migration so the dashboard knows about it, and use
`scheduler.jobs._run_tracker` so each pull lands a row in `ingest_runs`.
More work; only worth it once you've decided the source matters.

Below is **path A** — the right starting point for exploration.

In [ ]:
# Example: bring in a synthetic 'sentiment' dataset for our 3 symbols.
# In real life, replace with: API call to your data vendor, CSV upload,
# scrape, FRED/SEC EDGAR/etc.

rng = np.random.default_rng(42)
syms = ['AAPL', 'MSFT', 'NVDA']
dates = pd.date_range('2020-01-31', '2026-01-31', freq='ME')
rows = []
for d in dates:
    for s in syms:
        rows.append({'date': d.date(), 'symbol': s,
                     'sentiment_score': float(rng.normal(0.05 if s == 'NVDA' else 0.0, 0.3))})
sentiment = pd.DataFrame(rows)
sentiment.head()

In [ ]:
# Save it into the warehouse. `if_exists='replace'` is fine in research mode;
# use 'append' once you've stabilised the schema.
sentiment.to_sql('sentiment_monthly', con, if_exists='replace', index=False)

# Verify
pd.read_sql("SELECT * FROM sentiment_monthly LIMIT 5", con)

**For path B (tracked ingestion):** look at
`backend/scheduler/jobs.py`. Every job uses the `_run_tracker` context
manager which writes a row to `ingest_runs`. To add your own:

```python
from scheduler.jobs import _run_tracker

def ingest_my_source(start, end):
    with _run_tracker("my_data_source") as run:
        df = fetch_from_somewhere(start, end)
        # ... write to your table ...
        run.rows_ingested = len(df)
        run.summary = {"start": start, "end": end}
        return run.summary
```

Then add it to `JOBS = {...}` in the same file and it shows up in
`run_scheduled_ingest --job my_data_source` and on the `/schedules` page.

## 6. Computing custom features

Most of the time, **don't extend `fct_features`**. Just compute features
in pandas and use them in your run. Faster iteration, no migrations.

If a feature proves useful and you want it built in for everyone, port it
into `backend/features/indicators.py` (daily) or `backend/features/returns.py`
(monthly) and rerun `make build-features`.

Below: join our new sentiment data with the existing features.

In [ ]:
sentiment_db = pd.read_sql("SELECT * FROM sentiment_monthly", con, parse_dates=['date'])
merged = feats.merge(sentiment_db, on=['symbol', 'date'], how='inner')

# Add a derived feature on the fly: 3-month rolling mean of sentiment, per symbol.
merged = merged.sort_values(['symbol', 'date'])
merged['sentiment_3m'] = (
    merged.groupby('symbol')['sentiment_score']
          .rolling(3, min_periods=1).mean()
          .reset_index(level=0, drop=True)
)

merged[['symbol', 'date', 'sentiment_score', 'sentiment_3m', 'target_1m']].head()

## 7. A richer run

Same model, but adding the sentiment features. Multiple metrics,
feature importance, and we'll save a tiny artifact to disk.

In [ ]:
from pathlib import Path

RICH_FEATURES = FEATURES + ['sentiment_score', 'sentiment_3m']
df2 = merged.dropna(subset=RICH_FEATURES + ['target_1m']).sort_values('date')

split_at = df2['date'].quantile(0.8)
tr = df2[df2['date'] <= split_at]; te = df2[df2['date'] > split_at]

sc = StandardScaler().fit(tr[RICH_FEATURES])
Xtr = sc.transform(tr[RICH_FEATURES])
Xte = sc.transform(te[RICH_FEATURES])
model2 = Ridge(alpha=1.0, random_state=0).fit(Xtr, tr['target_1m'])
yp = model2.predict(Xte)

metrics = {
    'oos_mae': float(mean_absolute_error(te['target_1m'], yp)),
    'oos_r2':  float(r2_score(te['target_1m'], yp)),
    'ic':      float(np.corrcoef(te['target_1m'], yp)[0, 1]),
    'hit_rate_directional': float(((yp > 0) == (te['target_1m'] > 0)).mean()),
}
metrics

In [ ]:
# Save predictions as a small artifact under data/artifacts/<run_id>/
preds = pd.DataFrame({
    'date': te['date'].values,
    'symbol': te['symbol'].values,
    'y_true': te['target_1m'].values,
    'y_pred': yp,
})

with tracker.run(
    experiment='notebook_walkthrough',
    params={
        'model': 'ridge',
        'alpha': 1.0,
        'features': RICH_FEATURES,
        'data_sources': ['fct_features', 'sentiment_monthly'],
    },
    tags=['ridge', 'sentiment', 'notebook'],
    name='ridge_with_sentiment',
) as r:
    r.log_metrics(metrics)
    r.log_importance(dict(zip(RICH_FEATURES, np.abs(model2.coef_))))

    # Save predictions parquet alongside the run
    art_dir = DB.parent / 'artifacts' / r.run_id
    art_dir.mkdir(parents=True, exist_ok=True)
    preds_path = art_dir / 'predictions.parquet'
    preds.to_parquet(preds_path, index=False)
    r.log_artifact('predictions', str(preds_path), kind='parquet')

    rich_run_id = r.run_id
    print('run_id:', rich_run_id)

## 8. Querying run history from the notebook

Once you have a few runs you can compare them with SQL right here.
Useful for sweeps and ad-hoc analysis.

In [ ]:
# Most recent run per metric — ranked by ic.
pd.read_sql("""
  SELECT r.run_id, e.name AS experiment, r.name, r.status, r.started_at,
         MAX(CASE WHEN m.key = 'ic'      THEN m.value END) AS ic,
         MAX(CASE WHEN m.key = 'oos_mae' THEN m.value END) AS oos_mae,
         MAX(CASE WHEN m.key = 'oos_r2'  THEN m.value END) AS oos_r2
  FROM run r
  JOIN experiment e ON e.experiment_id = r.experiment_id
  LEFT JOIN run_metric m ON m.run_id = r.run_id
  WHERE e.name = 'notebook_walkthrough'
  GROUP BY r.run_id, e.name, r.name, r.status, r.started_at
  ORDER BY ic DESC NULLS LAST
""", con)

In [ ]:
# Pull every metric a specific run logged.
pd.read_sql(f"""
  SELECT key, value, ts FROM run_metric
  WHERE run_id = '{rich_run_id}' ORDER BY key
""", con)

## 9. Appending to a finished run with `tracker.attach`

Useful when you compute a follow-up analysis after a run already closed —
for example, a portfolio backtest on top of an RF run's picks. The
platform itself uses this internally (`portfolio_backtest` writes
`portfolio_*` metrics back onto the RF run that produced its picks).

In [ ]:
with tracker.attach(rich_run_id) as r:
    r.tag('reanalysed_in_notebook')
    r.log_metric('post_hoc_check', 0.123)
    r.param('reanalysed_at', pd.Timestamp.utcnow().isoformat())

print("appended. The run is still 'completed' — only its data grew.")

## 10. Workflow recap

**Daily loop:**
1. Open this notebook (or your own).
2. `make backend && make frontend` once, leave both running.
3. Iterate: edit cells, run them, log runs via `tracker.run(...)`.
4. Refresh the browser — leaderboard updates, run detail shows charts,
   compare two ridges side-by-side.

**When something is worth promoting from notebook to platform:**
- A reusable feature → port into `backend/features/indicators.py`,
  rerun `make build-features`.
- A reusable data source → add a job to `backend/scheduler/jobs.py` so
  it gets scheduled-ingest history.
- A reusable model → add to `backend/models_rf/` (or a new `models_*` app)
  with its own management command and result tables.

**Cheat-sheet of useful SQL queries:**
```sql
-- All metrics for a run
SELECT key, value FROM run_metric WHERE run_id = '...';

-- Best run per experiment
SELECT e.name, MAX(m.value) AS best_ic
FROM run r JOIN experiment e ON e.experiment_id = r.experiment_id
JOIN run_metric m ON m.run_id = r.run_id AND m.key = 'ic'
WHERE r.status = 'completed' GROUP BY e.name;

-- Feature importance averaged over completed runs
SELECT feature, AVG(importance) AS mean_imp, COUNT(*) AS n_runs
FROM run_feature_importance fi
JOIN run r ON r.run_id = fi.run_id AND r.status = 'completed'
GROUP BY feature ORDER BY mean_imp DESC;

-- Latest data freshness per symbol
SELECT symbol, MAX(date) FROM raw_prices GROUP BY symbol;
```